# GTSRB Top-1 — LnL-Ti

Bản tối giản của `Instructions_Top1_Improved_v2`: LnL-Ti, Mixup trước cooldown, checkpoint/resume và đánh giá Top-1.

In [ ]:
%pip install -q timm==0.9.16 einops scikit-learn tqdm

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

REPOSITORY_URL = 'https://github.com/Omid-Nejati/Locality-iN-Locality.git'

def is_repository(path: Path) -> bool:
    return (path / 'models' / 'localvit_tnt.py').exists()

override = os.environ.get('LNL_REPO_ROOT')
candidates = [Path(override).expanduser()] if override else []
for folder in (Path.cwd(), *Path.cwd().parents):
    candidates.extend([folder, folder / 'Locality-iN-Locality-main', folder / 'Locality-iN-Locality'])

repository_root = next((path.resolve() for path in candidates if is_repository(path)), None)
if repository_root is None:
    repository_root = Path('/content/Locality-iN-Locality') if Path('/content').exists() else Path.cwd() / 'Locality-iN-Locality'
    if not repository_root.exists():
        subprocess.run(['git', 'clone', REPOSITORY_URL, str(repository_root)], check=True)
    if not is_repository(repository_root):
        raise FileNotFoundError(f'LnL repository was not found at {repository_root}')

os.chdir(repository_root)
if str(repository_root) not in sys.path:
    sys.path.insert(0, str(repository_root))
print('Repository:', repository_root)

In [ ]:
import csv
import math
import random
import shutil
import zipfile
from collections import defaultdict
from urllib.request import urlretrieve

import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
import torchvision.transforms.functional as TF
from PIL import Image
from torch.utils.data import DataLoader, Subset
from tqdm.auto import tqdm

SEED = 42

def seed_everything(seed: int) -> None:
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

seed_everything(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
torch.backends.cudnn.benchmark = device.type == 'cuda'

print('PyTorch:', torch.__version__)
print('Torchvision:', torchvision.__version__)
print('Device:', device)

In [ ]:
DATA_DIR = Path('data')
DATA_DIR.mkdir(parents=True, exist_ok=True)
GTSRB_URLS = {
    'GTSRB_Final_Training_Images.zip': 'https://sid.erda.dk/public/archives/daaeac0d7ce1152aea9b61d9f1e19370/GTSRB_Final_Training_Images.zip',
    'GTSRB_Final_Test_Images.zip': 'https://sid.erda.dk/public/archives/daaeac0d7ce1152aea9b61d9f1e19370/GTSRB_Final_Test_Images.zip',
    'GTSRB_Final_Test_GT.zip': 'https://sid.erda.dk/public/archives/daaeac0d7ce1152aea9b61d9f1e19370/GTSRB_Final_Test_GT.zip',
}

def prepare_gtsrb() -> tuple[Path, Path]:
    for filename, url in GTSRB_URLS.items():
        archive_path = DATA_DIR / filename
        if not archive_path.exists():
            print('Downloading', filename)
            urlretrieve(url, archive_path)

    gtsrb_dir = DATA_DIR / 'GTSRB'
    if not (gtsrb_dir / 'Final_Training' / 'Images').exists():
        for filename in GTSRB_URLS:
            with zipfile.ZipFile(DATA_DIR / filename) as archive:
                archive.extractall(DATA_DIR)

    test_dir = gtsrb_dir / 'test'
    test_source = gtsrb_dir / 'Final_Test' / 'Images'
    if not test_dir.exists() or not any(test_dir.iterdir()):
        with (DATA_DIR / 'GT-final_test.csv').open(newline='') as csv_file:
            for row in csv.DictReader(csv_file, delimiter=';'):
                class_dir = test_dir / f"{int(row['ClassId']):04d}"
                class_dir.mkdir(parents=True, exist_ok=True)
                shutil.copy2(test_source / row['Filename'], class_dir / row['Filename'])
    return gtsrb_dir / 'Final_Training' / 'Images', test_dir

TRAIN_ROOT, TEST_ROOT = prepare_gtsrb()
print('Training data:', TRAIN_ROOT)
print('Test data:', TEST_ROOT)

In [ ]:
class ResizeWithPadding:
    def __init__(self, size: int = 224, fill: int = 128):
        self.size, self.fill = size, fill

    def __call__(self, image: Image.Image) -> Image.Image:
        width, height = image.size
        scale = self.size / max(width, height)
        new_width, new_height = max(1, round(width * scale)), max(1, round(height * scale))
        image = TF.resize(image, [new_height, new_width], interpolation=transforms.InterpolationMode.BICUBIC, antialias=True)
        horizontal, vertical = self.size - new_width, self.size - new_height
        return TF.pad(image, [horizontal // 2, vertical // 2, horizontal - horizontal // 2, vertical - vertical // 2], fill=self.fill)

ROI_PROBABILITY = 0.20
ROI_BORDER_RATIO = 0.08

def crop_expanded_roi(image: Image.Image, box: tuple[int, int, int, int], border_ratio: float) -> Image.Image:
    x1, y1, x2, y2 = box
    roi_width, roi_height = x2 - x1 + 1, y2 - y1 + 1
    pad_x, pad_y = round(roi_width * border_ratio), round(roi_height * border_ratio)
    left, top = max(0, x1 - pad_x), max(0, y1 - pad_y)
    right, bottom = min(image.width, x2 + pad_x + 1), min(image.height, y2 + pad_y + 1)
    if right <= left or bottom <= top:
        raise ValueError(f'Invalid ROI box: {box}')
    return image.crop((left, top, right, bottom))

def read_training_roi_boxes(root: Path) -> dict[str, tuple[int, int, int, int]]:
    boxes = {}
    for class_dir in sorted(path for path in root.iterdir() if path.is_dir()):
        annotation_path = class_dir / f'GT-{class_dir.name}.csv'
        with annotation_path.open(newline='') as csv_file:
            for row in csv.DictReader(csv_file, delimiter=';'):
                image_path = str((class_dir / row['Filename']).resolve())
                boxes[image_path] = tuple(int(row[key]) for key in ('Roi.X1', 'Roi.Y1', 'Roi.X2', 'Roi.Y2'))
    return boxes

class ROIAugmentedImageFolder(torchvision.datasets.ImageFolder):
    def __init__(self, root, *, roi_boxes, transform, roi_probability, border_ratio):
        super().__init__(root, transform=transform)
        self.roi_boxes = roi_boxes
        self.roi_probability = roi_probability
        self.border_ratio = border_ratio

    def __getitem__(self, index):
        path, target = self.samples[index]
        image = self.loader(path)
        if random.random() < self.roi_probability:
            image = crop_expanded_roi(
                image, self.roi_boxes[str(Path(path).resolve())], self.border_ratio
            )
        if self.transform is not None:
            image = self.transform(image)
        return image, target

NORMALIZE_MEAN = NORMALIZE_STD = (0.5, 0.5, 0.5)
train_transform = transforms.Compose([
    ResizeWithPadding(),
    transforms.RandomApply([transforms.RandomAffine(10, translate=(0.05, 0.05), scale=(0.90, 1.10), shear=3, interpolation=transforms.InterpolationMode.BILINEAR, fill=128)], p=0.70),
    transforms.RandomPerspective(distortion_scale=0.15, p=0.20, interpolation=transforms.InterpolationMode.BILINEAR, fill=128),
    transforms.RandomApply([transforms.ColorJitter(brightness=0.25, contrast=0.25, saturation=0.20, hue=0.02)], p=0.60),
    transforms.ToTensor(), transforms.Normalize(NORMALIZE_MEAN, NORMALIZE_STD),
])
eval_transform = transforms.Compose([ResizeWithPadding(), transforms.ToTensor(), transforms.Normalize(NORMALIZE_MEAN, NORMALIZE_STD)])

MICRO_BATCH_SIZE, EVAL_BATCH_SIZE, NUM_WORKERS = 16, 64, 0
VALIDATION_FRACTION = 0.20
index_dataset = torchvision.datasets.ImageFolder(TRAIN_ROOT)
groups_by_class = defaultdict(lambda: defaultdict(list))
for sample_index, (path, class_id) in enumerate(index_dataset.samples):
    sequence_id = Path(path).stem.split('_')[0]
    groups_by_class[class_id][sequence_id].append(sample_index)

rng = np.random.default_rng(SEED)
train_indices, validation_indices = [], []
for class_id, sequences in groups_by_class.items():
    class_groups = list(sequences.items())
    rng.shuffle(class_groups)
    target_count = round(sum(len(indices) for _, indices in class_groups) * VALIDATION_FRACTION)
    selected_count = 0
    while len(class_groups) > 1 and selected_count < target_count:
        position = min(range(len(class_groups)), key=lambda i: abs(selected_count + len(class_groups[i][1]) - target_count))
        _, indices = class_groups.pop(position)
        validation_indices.extend(indices)
        selected_count += len(indices)
    train_indices.extend(index for _, indices in class_groups for index in indices)

assert set(train_indices).isdisjoint(validation_indices)
TRAINING_ROI_BOXES = read_training_roi_boxes(TRAIN_ROOT)
if len(TRAINING_ROI_BOXES) != len(index_dataset):
    raise RuntimeError(f'ROI annotations: {len(TRAINING_ROI_BOXES)}; training images: {len(index_dataset)}')
training_dataset = ROIAugmentedImageFolder(
    TRAIN_ROOT,
    roi_boxes=TRAINING_ROI_BOXES,
    transform=train_transform,
    roi_probability=ROI_PROBABILITY,
    border_ratio=ROI_BORDER_RATIO,
)
validation_dataset = torchvision.datasets.ImageFolder(TRAIN_ROOT, transform=eval_transform)
testset = torchvision.datasets.ImageFolder(TEST_ROOT, transform=eval_transform)
trainset, validationset = Subset(training_dataset, train_indices), Subset(validation_dataset, validation_indices)

def make_loader(dataset, batch_size, *, shuffle=False, drop_last=False):
    return DataLoader(dataset, batch_size=batch_size, shuffle=shuffle, num_workers=NUM_WORKERS, pin_memory=device.type == 'cuda', persistent_workers=NUM_WORKERS > 0, drop_last=drop_last)

train_loader = make_loader(trainset, MICRO_BATCH_SIZE, shuffle=True, drop_last=True)
validation_loader = make_loader(validationset, EVAL_BATCH_SIZE)
test_loader = make_loader(testset, EVAL_BATCH_SIZE)
NUM_CLASSES = len(index_dataset.classes)
print(f'Train: {len(trainset)} | Validation: {len(validationset)} | Test: {len(testset)} | Classes: {NUM_CLASSES}')
print(f'Training-only ROI crop: p={ROI_PROBABILITY:.2f}, border={ROI_BORDER_RATIO:.2f}; validation/test remain clean.')

## LnL-Ti, optimizer, scheduler và checkpoint

In [ ]:
from LNL import LNL_Ti

model = LNL_Ti(pretrained=False, num_classes=NUM_CLASSES, drop_path_rate=0.10).to(device)
print(f'LnL-Ti parameters: {sum(parameter.numel() for parameter in model.parameters()):,}')

NUM_EPOCHS, MIN_EPOCHS = 100, 40
EARLY_STOP_PATIENCE, EARLY_STOP_MIN_DELTA = 15, 0.02
TARGET_VALIDATION_TOP1, TARGET_STABLE_EPOCHS = 99.5, 3
WARMUP_EPOCHS, COOLDOWN_START_EPOCH = 5, 31  # Human-readable epoch numbers; cooldown begins at epoch 31.
ACCUMULATION_STEPS = 4
LEARNING_RATE, MIN_LR_FACTOR, WEIGHT_DECAY = 5e-4, 0.01, 0.05
MIXUP_PROBABILITY, MIXUP_ALPHA = 0.35, 0.25

checkpoint_dir_override = os.environ.get('CHECKPOINT_DIR')
colab_drive_dir = Path('/content/drive/MyDrive')
CHECKPOINT_DIR = Path(checkpoint_dir_override).expanduser() if checkpoint_dir_override else (colab_drive_dir / 'LNL_Ti_checkpoints' if colab_drive_dir.exists() else Path.cwd() / 'checkpoints')
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINT_PATH = CHECKPOINT_DIR / 'LNL_Ti_GTSRB_v2_roi_mixup_cooldown.pth'
RESUME_TRAINING = os.environ.get('RESUME_TRAINING', '0').strip().lower() in {'1', 'true', 'yes', 'on'}

criterion = nn.CrossEntropyLoss(label_smoothing=0.02)
no_decay, decay = [], []
for name, parameter in model.named_parameters():
    if parameter.requires_grad:
        (no_decay if parameter.ndim == 1 or name.endswith('.bias') or name in {'cls_token', 'patch_pos', 'pixel_pos'} else decay).append(parameter)
optimizer = optim.AdamW([{'params': decay, 'weight_decay': WEIGHT_DECAY}, {'params': no_decay, 'weight_decay': 0.0}], lr=LEARNING_RATE, betas=(0.9, 0.999))

def learning_rate_factor(epoch: int) -> float:
    if epoch < WARMUP_EPOCHS:
        return (epoch + 1) / WARMUP_EPOCHS
    progress = (epoch - WARMUP_EPOCHS) / max(1, NUM_EPOCHS - WARMUP_EPOCHS)
    return MIN_LR_FACTOR + (1 - MIN_LR_FACTOR) * 0.5 * (1 + math.cos(math.pi * progress))

scheduler = optim.lr_scheduler.LambdaLR(optimizer, lr_lambda=learning_rate_factor)
scaler = torch.cuda.amp.GradScaler(enabled=device.type == 'cuda')
print('Checkpoint:', CHECKPOINT_PATH)

In [ ]:
@torch.inference_mode()
def calculate_top1(model: nn.Module, loader: DataLoader) -> float:
    model.eval()
    correct = total = 0
    for images, labels in loader:
        images, labels = images.to(device, non_blocking=True), labels.to(device, non_blocking=True)
        with torch.cuda.amp.autocast(enabled=device.type == 'cuda'):
            predictions = model(images).argmax(dim=1)
        correct += predictions.eq(labels).sum().item()
        total += labels.size(0)
    return 100 * correct / total

def cpu_state_dict(module: nn.Module) -> dict:
    return {name: value.detach().cpu().clone() for name, value in module.state_dict().items()}

def save_checkpoint(next_epoch, best_top1, best_state, plateau_epochs, target_streak):
    payload = {
        'format_version': 1, 'epoch': next_epoch, 'best_validation_top1': best_top1,
        'model_state_dict': model.state_dict(), 'best_model_state_dict': best_state or cpu_state_dict(model),
        'optimizer_state_dict': optimizer.state_dict(), 'scheduler_state_dict': scheduler.state_dict(),
        'scaler_state_dict': scaler.state_dict(), 'plateau_epochs': plateau_epochs, 'target_streak': target_streak,
        'rng_state': torch.get_rng_state(), 'numpy_rng_state': np.random.get_state(), 'python_rng_state': random.getstate(),
    }
    if torch.cuda.is_available():
        payload['cuda_rng_state_all'] = torch.cuda.get_rng_state_all()
    temporary_path = CHECKPOINT_PATH.with_suffix(CHECKPOINT_PATH.suffix + '.tmp')
    torch.save(payload, temporary_path)
    os.replace(temporary_path, CHECKPOINT_PATH)

def load_torch_checkpoint(path: Path):
    try:
        return torch.load(path, map_location=device, weights_only=False)
    except TypeError:  # PyTorch versions before the weights_only argument.
        return torch.load(path, map_location=device)

def load_checkpoint():
    if not RESUME_TRAINING or not CHECKPOINT_PATH.exists():
        return 0, -1.0, None, 0, 0
    checkpoint = load_torch_checkpoint(CHECKPOINT_PATH)
    model.load_state_dict(checkpoint['model_state_dict'], strict=True)
    optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
    scheduler.load_state_dict(checkpoint['scheduler_state_dict'])
    scaler.load_state_dict(checkpoint.get('scaler_state_dict', {}))
    for state in optimizer.state.values():
        for name, value in state.items():
            if torch.is_tensor(value):
                state[name] = value.to(device)
    torch.set_rng_state(checkpoint['rng_state'].cpu())
    np.random.set_state(checkpoint['numpy_rng_state'])
    random.setstate(checkpoint['python_rng_state'])
    if torch.cuda.is_available() and 'cuda_rng_state_all' in checkpoint:
        torch.cuda.set_rng_state_all(checkpoint['cuda_rng_state_all'])
    print(f"Resumed at epoch {checkpoint['epoch']}/{NUM_EPOCHS}; best validation Top-1: {checkpoint['best_validation_top1']:.3f}%")
    return checkpoint['epoch'], checkpoint['best_validation_top1'], checkpoint.get('best_model_state_dict'), checkpoint.get('plateau_epochs', 0), checkpoint.get('target_streak', 0)

In [ ]:
def mixup_probability(epoch_number: int) -> float:
    return MIXUP_PROBABILITY if epoch_number < COOLDOWN_START_EPOCH else 0.0

def train_one_epoch(epoch: int) -> float:
    model.train()
    optimizer.zero_grad(set_to_none=True)
    running_loss = 0.0
    epoch_number = epoch + 1
    current_mixup_probability = mixup_probability(epoch_number)
    progress_bar = tqdm(
        train_loader,
        total=len(train_loader),
        desc=f'Epoch {epoch_number:03d}/{NUM_EPOCHS}',
        unit='batch',
        dynamic_ncols=True,
        leave=True,
    )
    for step, (images, labels) in enumerate(progress_bar, start=1):
        images, labels = images.to(device, non_blocking=True), labels.to(device, non_blocking=True)
        use_mixup = images.size(0) > 1 and np.random.random() < current_mixup_probability
        with torch.cuda.amp.autocast(enabled=device.type == 'cuda'):
            if use_mixup:
                lam = max(float(np.random.beta(MIXUP_ALPHA, MIXUP_ALPHA)), 0.5)
                permutation = torch.randperm(images.size(0), device=device)
                logits = model(lam * images + (1 - lam) * images[permutation])
                loss = lam * criterion(logits, labels) + (1 - lam) * criterion(logits, labels[permutation])
            else:
                loss = criterion(model(images), labels)
        running_loss += loss.detach().item()
        if step == 1 or step % 10 == 0 or step == len(train_loader):
            progress_bar.set_postfix(
                loss=f'{running_loss / step:.4f}',
                mixup=f'{current_mixup_probability:.2f}',
                lr=f"{optimizer.param_groups[0]['lr']:.2e}",
            )
        scaler.scale(loss / ACCUMULATION_STEPS).backward()
        if step % ACCUMULATION_STEPS == 0 or step == len(train_loader):
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad(set_to_none=True)
    return running_loss / max(1, len(train_loader))

start_epoch, best_validation_top1, best_model_state_dict, plateau_epochs, target_streak = load_checkpoint()
for epoch in range(start_epoch, NUM_EPOCHS):
    training_loss = train_one_epoch(epoch)
    validation_top1 = calculate_top1(model, validation_loader)
    improved = validation_top1 > best_validation_top1 + EARLY_STOP_MIN_DELTA
    if validation_top1 > best_validation_top1:
        best_validation_top1, best_model_state_dict = validation_top1, cpu_state_dict(model)
    plateau_epochs = 0 if improved else plateau_epochs + 1
    target_streak = target_streak + 1 if validation_top1 > TARGET_VALIDATION_TOP1 else 0
    scheduler.step()
    save_checkpoint(epoch + 1, best_validation_top1, best_model_state_dict, plateau_epochs, target_streak)
    current_mixup_probability = mixup_probability(epoch + 1)
    in_cooldown = epoch + 1 >= COOLDOWN_START_EPOCH
    phase = 'cooldown (clean)' if in_cooldown else f'mixup p={current_mixup_probability:.2f}'
    print(f'Epoch {epoch + 1:03d}/{NUM_EPOCHS} | {phase} | label smoothing: 0.02 | loss: {training_loss:.5f} | validation Top-1: {validation_top1:.3f}% | best: {best_validation_top1:.3f}% | plateau: {plateau_epochs}/{EARLY_STOP_PATIENCE}')
    if epoch + 1 >= MIN_EPOCHS and (target_streak >= TARGET_STABLE_EPOCHS or plateau_epochs >= EARLY_STOP_PATIENCE):
        print('Early stopping triggered.')
        break

print(f'Best validation Top-1: {best_validation_top1:.3f}%')

In [ ]:
checkpoint = load_torch_checkpoint(CHECKPOINT_PATH)
model.load_state_dict(checkpoint.get('best_model_state_dict', checkpoint['model_state_dict']), strict=True)
test_top1 = calculate_top1(model, test_loader)
print(f'Clean Test Top-1 accuracy: {test_top1:.3f}%')

try:
    from google.colab import files
    files.download(str(CHECKPOINT_PATH))
except ImportError:
    print(f'Checkpoint saved at: {CHECKPOINT_PATH.resolve()}')